In [ ]:
!pip install timm kornia torchvision matplotlib opencv-python

In [ ]:
# ------------------------------
# 0. Mount Google Drive
# ------------------------------
from google.colab import drive
drive.mount('/content/drive')

# ------------------------------
# 1. Import Libraries
# ------------------------------
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import Compose, Resize, ToTensor, RandomHorizontalFlip, RandomRotation
import torchvision
import timm  # For Swin Transformer
import kornia
import kornia.losses
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import os
import torch.nn.functional as F

# ------------------------------
# 2. (Optional) Synthetic Data Generation
# ------------------------------
def generate_synthetic_pair(save_path=None):
    """Generates a synthetic CT-MRI pair with tumor simulation."""
    ct = np.zeros((256, 256), dtype=np.uint8)
    cv2.circle(ct, (128, 128), 40, 200, -1)  # Tumor core
    cv2.circle(ct, (128, 128), 60, 150, 10)   # Tumor boundary
    ct = np.clip(ct + np.random.normal(0, 30, (256, 256)), 0, 255).astype(np.uint8)

    mri = cv2.GaussianBlur(ct, (61, 61), 0)  # Simulate soft tissue appearance
    mri = cv2.adaptiveThreshold(mri, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                cv2.THRESH_BINARY, 101, 5)  # Enhance boundaries

    if save_path:
        os.makedirs(os.path.join(save_path, "ct"), exist_ok=True)
        os.makedirs(os.path.join(save_path, "mri"), exist_ok=True)
        Image.fromarray(ct).save(f"{save_path}/ct/ct_{np.random.randint(1000):04d}.png")
        Image.fromarray(mri).save(f"{save_path}/mri/mri_{np.random.randint(1000):04d}.png")
    return ct, mri

# Generate synthetic dataset if not already present (optional)
if not os.path.exists("enhanced_dataset/ct") or len(os.listdir("enhanced_dataset/ct")) == 0:
    os.makedirs("enhanced_dataset/ct", exist_ok=True)
    os.makedirs("enhanced_dataset/mri", exist_ok=True)
    for _ in range(500):  # Generate 500 pairs
        generate_synthetic_pair("enhanced_dataset")

# ------------------------------
# 3. SwinUNet Architecture with Skip Connections and Dimension Handling
# ------------------------------
class SwinUNet(nn.Module):
    def __init__(self):
        super().__init__()
        # Use Swin Transformer Tiny as encoder with features_only=True
        self.encoder = timm.create_model(
            "swin_tiny_patch4_window7_224",
            pretrained=True,
            in_chans=1,
            features_only=True
        )
        # Expected feature channels: [96, 192, 384, 768] for Swin Tiny
        self.feature_channels = [96, 192, 384, 768]

        # Define decoder with skip connections
        self.up1 = nn.ConvTranspose2d(768, 384, kernel_size=4, stride=2, padding=1)
        self.conv1 = nn.Sequential(
            nn.ReLU(),
            nn.Conv2d(384 + 384, 384, kernel_size=3, padding=1),
            nn.ReLU()
        )
        self.up2 = nn.ConvTranspose2d(384, 192, kernel_size=4, stride=2, padding=1)
        self.conv2 = nn.Sequential(
            nn.ReLU(),
            nn.Conv2d(192 + 192, 192, kernel_size=3, padding=1),
            nn.ReLU()
        )
        self.up3 = nn.ConvTranspose2d(192, 96, kernel_size=4, stride=2, padding=1)
        self.conv3 = nn.Sequential(
            nn.ReLU(),
            nn.Conv2d(96 + 96, 96, kernel_size=3, padding=1),
            nn.ReLU()
        )
        # Additional upsampling layers to reach 224x224
        self.up4 = nn.ConvTranspose2d(96, 48, kernel_size=4, stride=2, padding=1)
        self.conv4 = nn.Sequential(
            nn.ReLU(),
            nn.Conv2d(48, 48, kernel_size=3, padding=1),
            nn.ReLU()
        )
        self.up5 = nn.ConvTranspose2d(48, 24, kernel_size=4, stride=2, padding=1)
        self.conv5 = nn.Sequential(
            nn.ReLU(),
            nn.Conv2d(24, 24, kernel_size=3, padding=1),
            nn.ReLU()
        )
        self.final_conv = nn.Sequential(
            nn.Conv2d(24, 1, kernel_size=3, padding=1),
            nn.Sigmoid()
        )

    def _maybe_permute(self, feature, expected_channels):
        # Ensure features are in (N, C, H, W). If in channel-last format, permute.
        if feature.shape[-1] == expected_channels:
            return feature.permute(0, 3, 1, 2)
        else:
            return feature

    def forward(self, x):
        features = self.encoder(x)  # Multi-scale features from encoder

        # Convert features to (N, C, H, W) as needed:
        f4 = self._maybe_permute(features[-1], self.feature_channels[-1])  # 768 channels
        f3 = self._maybe_permute(features[-2], self.feature_channels[-2])  # 384 channels
        f2 = self._maybe_permute(features[-3], self.feature_channels[-3])  # 192 channels
        f1 = self._maybe_permute(features[-4], self.feature_channels[-4])  # 96 channels

        # Decoder with skip connections:
        x = self.up1(f4)
        if x.shape[-2:] != f3.shape[-2:]:
            x = nn.functional.interpolate(x, size=f3.shape[-2:], mode='bilinear', align_corners=False)
        x = self.conv1(torch.cat([x, f3], dim=1))

        x = self.up2(x)
        if x.shape[-2:] != f2.shape[-2:]:
            x = nn.functional.interpolate(x, size=f2.shape[-2:], mode='bilinear', align_corners=False)
        x = self.conv2(torch.cat([x, f2], dim=1))

        x = self.up3(x)
        if x.shape[-2:] != f1.shape[-2:]:
            x = nn.functional.interpolate(x, size=f1.shape[-2:], mode='bilinear', align_corners=False)
        x = self.conv3(torch.cat([x, f1], dim=1))

        x = self.up4(x)
        x = self.conv4(x)
        x = self.up5(x)
        x = self.conv5(x)

        return self.final_conv(x)

# ------------------------------
# 4. Dataset Class (CT & MRI)
# ------------------------------
class CTMRIDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        """
        root_dir should be the path to either the 'train' or 'test' folder.
        Each folder must contain two subfolders: "CT" and "MRI".
        """
        self.ct_dir = os.path.join(root_dir, "CT")
        self.mri_dir = os.path.join(root_dir, "MRI")
        self.transform = transform
        self.ct_files = sorted(os.listdir(self.ct_dir))
        self.mri_files = sorted(os.listdir(self.mri_dir))

        if len(self.ct_files) == 0 or len(self.mri_files) == 0:
            raise ValueError("CT or MRI directories are empty. Please ensure the dataset is correctly populated.")

    def __len__(self):
        return len(self.ct_files)

    def __getitem__(self, idx):
        ct = Image.open(os.path.join(self.ct_dir, self.ct_files[idx])).convert("L")
        mri = Image.open(os.path.join(self.mri_dir, self.mri_files[idx])).convert("L")
        if self.transform:
            ct = self.transform(ct)
            mri = self.transform(mri)
        return ct, mri

# ------------------------------
# 5. Hybrid Loss Function
# ------------------------------
class HybridLoss(nn.Module):
    def __init__(self):
        super().__init__()
        # Use VGG16 features (first 16 layers) for perceptual loss
        self.vgg = torchvision.models.vgg16(weights=torchvision.models.VGG16_Weights.IMAGENET1K_V1).features[:16]
        for param in self.vgg.parameters():
            param.requires_grad = False
        self.ssim_loss_fn = kornia.losses.SSIMLoss(window_size=11)

    def forward(self, pred, target):
        # Structural similarity loss: 1 - SSIM
        ssim_loss = 1 - self.ssim_loss_fn(pred, target)

        # Perceptual loss using VGG16 (repeat grayscale to 3 channels)
        pred_features = self.vgg(pred.repeat(1, 3, 1, 1))
        target_features = self.vgg(target.repeat(1, 3, 1, 1))
        perceptual_loss = nn.MSELoss()(pred_features, target_features)

        # Total variation loss (averaged over the batch)
        tv_loss = kornia.losses.total_variation(pred).mean()

        return 0.6 * ssim_loss + 0.3 * perceptual_loss + 0.1 * tv_loss

# ------------------------------
# 6. Training Function with Evaluation & Image Saving
# ------------------------------
def train():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # Data augmentation for training
    transform = Compose([
        Resize((224, 224)),
        ToTensor(),
        RandomHorizontalFlip(),
        RandomRotation(10)
    ])
    #/content/drive/MyDrive/Final Year Project (FYP)/Datasets/1/CT-MRI Dataset/CT
    train_dir = "/content/drive/MyDrive/Final Year Project (FYP)/Datasets/1/CT-MRI Train Test Split Dataset/CT-MRI/train"
    test_dir = "/content/drive/MyDrive/Final Year Project (FYP)/Datasets/1/CT-MRI Train Test Split Dataset/CT-MRI/test"

    train_dataset = CTMRIDataset(train_dir, transform=transform)
    test_dataset = CTMRIDataset(test_dir, transform=Compose([Resize((224, 224)), ToTensor()]))
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=4)

    model = SwinUNet().to(device)
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    # Use ReduceLROnPlateau scheduler based on average loss
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.1, verbose=True)
    criterion = HybridLoss().to(device)

    # Directory for saving generated images
    save_img_dir = "/content/drive/MyDrive/Final Year Project (FYP)/Generated_Images"
    os.makedirs(save_img_dir, exist_ok=True)

    for epoch in range(50):
        model.train()
        epoch_loss = 0
        for ct, mri in train_loader:
            ct, mri = ct.to(device), mri.to(device)
            optimizer.zero_grad()
            output = model(ct)
            loss = criterion(output, mri)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        avg_loss = epoch_loss / len(train_loader)
        scheduler.step(avg_loss)
        print(f"Epoch {epoch+1}/50 - Loss: {avg_loss:.4f}")

        # Evaluate one sample from test dataset at each epoch:
        model.eval()
        with torch.no_grad():
            test_ct, test_mri = test_dataset[0]
            test_ct_tensor = test_ct.unsqueeze(0).to(device)
            gen_mri = model(test_ct_tensor).squeeze().cpu().numpy()

            # Compute additional metrics:
            gen_mri_tensor = torch.tensor(gen_mri).unsqueeze(0).unsqueeze(0)
            mri_tensor = test_mri.unsqueeze(0)
            psnr_val = kornia.metrics.psnr(gen_mri_tensor, mri_tensor, max_val=1.0)
            ssim_val = kornia.metrics.ssim(gen_mri_tensor, mri_tensor, window_size=11)
            mse_val = nn.MSELoss()(gen_mri_tensor, mri_tensor).mean()

            print(f"Epoch {epoch+1} Metrics -> PSNR: {psnr_val.item():.2f} dB, SSIM: {ssim_val.item():.4f}, MSE: {mse_val.item():.4f}")

            # Plot one generated MRI and its paired ground truth
            fig, ax = plt.subplots(1, 3, figsize=(15, 5))
            ax[0].imshow(test_ct.squeeze(), cmap='gray')
            ax[0].set_title("Input CT")
            ax[1].imshow(gen_mri, cmap='gray')
            ax[1].set_title("Generated MRI")
            ax[2].imshow(test_mri.squeeze(), cmap='gray')
            ax[2].set_title("Ground Truth MRI")
            plt.show()

            # Save generated images every 5 epochs:
            if (epoch+1) % 5 == 0:
                save_path = os.path.join(save_img_dir, f"epoch_{epoch+1}")
                os.makedirs(save_path, exist_ok=True)
                Image.fromarray((test_ct.squeeze().cpu().numpy()*255).astype(np.uint8)).save(os.path.join(save_path, "input_ct.png"))
                Image.fromarray((gen_mri*255).astype(np.uint8)).save(os.path.join(save_path, "generated_mri.png"))
                Image.fromarray((test_mri.squeeze().cpu().numpy()*255).astype(np.uint8)).save(os.path.join(save_path, "ground_truth_mri.png"))
                print(f"Saved generated images for epoch {epoch+1} to {save_path}")

        # Save the model every 5 epochs
        if (epoch+1) % 5 == 0:
            model_save_path = f"/content/drive/MyDrive/Final Year Project (FYP)/swinunet_epoch{epoch+1}.pth"
            torch.save(model.state_dict(), model_save_path)
            print(f"Model saved to {model_save_path}")

def evaluate(model_path):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    transform = Compose([Resize((224, 224)), ToTensor()])
    test_dir = "/content/drive/MyDrive/Final Year Project (FYP)/Datasets/1/CT-MRI Train Test Split Dataset/CT-MRI/test"
    test_dataset = CTMRIDataset(test_dir, transform=transform)
    model = SwinUNet().to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    ct, mri = test_dataset[0]
    ct_tensor = ct.unsqueeze(0).to(device)

    with torch.no_grad():
        gen_mri = model(ct_tensor).squeeze().cpu().numpy()

    gen_mri_tensor = torch.tensor(gen_mri).unsqueeze(0).unsqueeze(0)
    mri_tensor = mri.unsqueeze(0)

    psnr_val = kornia.metrics.psnr(gen_mri_tensor, mri_tensor, max_val=1.0)
    ssim_val = kornia.metrics.ssim(gen_mri_tensor, mri_tensor, window_size=11)
    mse_val = nn.MSELoss()(gen_mri_tensor, mri_tensor).mean()

    print(f"Evaluation Metrics -> PSNR: {psnr_val.item():.2f} dB, SSIM: {ssim_val.item():.4f}, MSE: {mse_val.item():.4f}")

    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    ax[0].imshow(ct.squeeze(), cmap='gray')
    ax[0].set_title("Input CT")
    ax[1].imshow(gen_mri, cmap='gray')
    ax[1].set_title(f"Generated MRI\nPSNR: {psnr_val:.2f} dB, SSIM: {ssim_val:.4f}")
    ax[2].imshow(mri.squeeze(), cmap='gray')
    ax[2].set_title("Ground Truth MRI")
    plt.show()

if __name__ == "__main__":
    train()
    # After training, update the model path if needed.
    evaluate("/content/drive/MyDrive/Final Year Project (FYP)/swinunet_epoch50.pth")


In [ ]:
# Install dependencies if needed:
# !pip install timm kornia torchvision matplotlib opencv-python

# ------------------------------
# 0. Mount Google Drive
# ------------------------------
from google.colab import drive
drive.mount('/content/drive')

# ------------------------------
# 1. Import Libraries
# ------------------------------
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import Compose, Resize, ToTensor, RandomHorizontalFlip, RandomRotation
import torchvision
import timm  # For Swin Transformer
import kornia
import kornia.losses
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import os
import torch.nn.functional as F

# ------------------------------
# 2. (Optional) Synthetic Data Generation
# ------------------------------
def generate_synthetic_pair(save_path=None):
    """Generates a synthetic CT-MRI pair with tumor simulation."""
    ct = np.zeros((256, 256), dtype=np.uint8)
    cv2.circle(ct, (128, 128), 40, 200, -1)  # Tumor core
    cv2.circle(ct, (128, 128), 60, 150, 10)   # Tumor boundary
    ct = np.clip(ct + np.random.normal(0, 30, (256, 256)), 0, 255).astype(np.uint8)

    mri = cv2.GaussianBlur(ct, (61, 61), 0)  # Simulate soft tissue appearance
    mri = cv2.adaptiveThreshold(mri, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                cv2.THRESH_BINARY, 101, 5)  # Enhance boundaries

    if save_path:
        os.makedirs(os.path.join(save_path, "ct"), exist_ok=True)
        os.makedirs(os.path.join(save_path, "mri"), exist_ok=True)
        Image.fromarray(ct).save(f"{save_path}/ct/ct_{np.random.randint(1000):04d}.png")
        Image.fromarray(mri).save(f"{save_path}/mri/mri_{np.random.randint(1000):04d}.png")
    return ct, mri

# Generate synthetic dataset if not already present
if not os.path.exists("enhanced_dataset/ct") or len(os.listdir("enhanced_dataset/ct")) == 0:
    os.makedirs("enhanced_dataset/ct", exist_ok=True)
    os.makedirs("enhanced_dataset/mri", exist_ok=True)
    for _ in range(500):  # Generate 500 pairs
        generate_synthetic_pair("enhanced_dataset")

# ------------------------------
# 3. SwinUNet Architecture with Skip Connections and Dimension Handling
# ------------------------------
class SwinUNet(nn.Module):
    def __init__(self):
        super().__init__()
        # Use Swin Transformer Tiny as encoder with features_only=True
        self.encoder = timm.create_model(
            "swin_tiny_patch4_window7_224",
            pretrained=True,
            in_chans=1,
            features_only=True
        )
        # Expected feature channels: [96, 192, 384, 768] for Swin Tiny
        self.feature_channels = [96, 192, 384, 768]

        # Define decoder with skip connections
        self.up1 = nn.ConvTranspose2d(768, 384, kernel_size=4, stride=2, padding=1)
        self.conv1 = nn.Sequential(
            nn.ReLU(),
            nn.Conv2d(384 + 384, 384, kernel_size=3, padding=1),
            nn.ReLU()
        )
        self.up2 = nn.ConvTranspose2d(384, 192, kernel_size=4, stride=2, padding=1)
        self.conv2 = nn.Sequential(
            nn.ReLU(),
            nn.Conv2d(192 + 192, 192, kernel_size=3, padding=1),
            nn.ReLU()
        )
        self.up3 = nn.ConvTranspose2d(192, 96, kernel_size=4, stride=2, padding=1)
        self.conv3 = nn.Sequential(
            nn.ReLU(),
            nn.Conv2d(96 + 96, 96, kernel_size=3, padding=1),
            nn.ReLU()
        )
        # Additional upsampling layers to reach 224x224
        self.up4 = nn.ConvTranspose2d(96, 48, kernel_size=4, stride=2, padding=1)
        self.conv4 = nn.Sequential(
            nn.ReLU(),
            nn.Conv2d(48, 48, kernel_size=3, padding=1),
            nn.ReLU()
        )
        self.up5 = nn.ConvTranspose2d(48, 24, kernel_size=4, stride=2, padding=1)
        self.conv5 = nn.Sequential(
            nn.ReLU(),
            nn.Conv2d(24, 24, kernel_size=3, padding=1),
            nn.ReLU()
        )
        self.final_conv = nn.Sequential(
            nn.Conv2d(24, 1, kernel_size=3, padding=1),
            nn.Sigmoid()
        )

    def _maybe_permute(self, feature, expected_channels):
        # Ensure features are in (N, C, H, W). If in channel-last format, permute.
        if feature.shape[-1] == expected_channels:
            return feature.permute(0, 3, 1, 2)
        else:
            return feature

    def forward(self, x):
        features = self.encoder(x)  # Multi-scale features from encoder

        # Convert features to (N, C, H, W) as needed:
        f4 = self._maybe_permute(features[-1], self.feature_channels[-1])  # 768 channels
        f3 = self._maybe_permute(features[-2], self.feature_channels[-2])  # 384 channels
        f2 = self._maybe_permute(features[-3], self.feature_channels[-3])  # 192 channels
        f1 = self._maybe_permute(features[-4], self.feature_channels[-4])  # 96 channels

        # Decoder with skip connections:
        x = self.up1(f4)
        if x.shape[-2:] != f3.shape[-2:]:
            x = nn.functional.interpolate(x, size=f3.shape[-2:], mode='bilinear', align_corners=False)
        x = self.conv1(torch.cat([x, f3], dim=1))

        x = self.up2(x)
        if x.shape[-2:] != f2.shape[-2:]:
            x = nn.functional.interpolate(x, size=f2.shape[-2:], mode='bilinear', align_corners=False)
        x = self.conv2(torch.cat([x, f2], dim=1))

        x = self.up3(x)
        if x.shape[-2:] != f1.shape[-2:]:
            x = nn.functional.interpolate(x, size=f1.shape[-2:], mode='bilinear', align_corners=False)
        x = self.conv3(torch.cat([x, f1], dim=1))

        x = self.up4(x)
        x = self.conv4(x)
        x = self.up5(x)
        x = self.conv5(x)

        return self.final_conv(x)

# ------------------------------
# 4. Dataset Class (CT & MRI)
# ------------------------------
class CTMRIDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        """
        root_dir should be the path to either the 'train' or 'test' folder.
        Each folder must contain two subfolders: "CT" and "MRI".
        """
        self.ct_dir = os.path.join(root_dir, "CT")
        self.mri_dir = os.path.join(root_dir, "MRI")
        self.transform = transform
        self.ct_files = sorted(os.listdir(self.ct_dir))
        self.mri_files = sorted(os.listdir(self.mri_dir))

        if len(self.ct_files) == 0 or len(self.mri_files) == 0:
            raise ValueError("CT or MRI directories are empty. Please ensure the dataset is correctly populated.")

    def __len__(self):
        return len(self.ct_files)

    def __getitem__(self, idx):
        ct = Image.open(os.path.join(self.ct_dir, self.ct_files[idx])).convert("L")
        mri = Image.open(os.path.join(self.mri_dir, self.mri_files[idx])).convert("L")
        if self.transform:
            ct = self.transform(ct)
            mri = self.transform(mri)
        return ct, mri

# ------------------------------
# 5. Hybrid Loss Function
# ------------------------------
class HybridLoss(nn.Module):
    def __init__(self):
        super().__init__()
        # Use VGG16 features (first 16 layers) for perceptual loss.
        self.vgg = torchvision.models.vgg16(weights=torchvision.models.VGG16_Weights.IMAGENET1K_V1).features[:16]
        for param in self.vgg.parameters():
            param.requires_grad = False
        self.ssim_loss_fn = kornia.losses.SSIMLoss(window_size=11)

    def forward(self, pred, target):
        # Structural similarity loss: 1 - SSIM
        ssim_loss = 1 - self.ssim_loss_fn(pred, target)

        # Perceptual loss using VGG16 (repeat grayscale to 3 channels)
        pred_features = self.vgg(pred.repeat(1, 3, 1, 1))
        target_features = self.vgg(target.repeat(1, 3, 1, 1))
        perceptual_loss = nn.MSELoss()(pred_features, target_features)

        # Total variation loss (averaged over the batch)
        tv_loss = kornia.losses.total_variation(pred).mean()

        return 0.6 * ssim_loss + 0.3 * perceptual_loss + 0.1 * tv_loss

# ------------------------------
# 6. Training Function with ReduceLROnPlateau and Saving Every 5 Epochs
# ------------------------------
def train():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # Use data augmentation to help generalization.
    transform = Compose([Resize((224, 224)), ToTensor(), RandomHorizontalFlip(), RandomRotation(10)])
    train_dir = "/content/drive/MyDrive/FYP (Final Year Project)/Datasets/1/CT-MRI Train Test Split Dataset/CT-MRI/train"
    train_dataset = CTMRIDataset(train_dir, transform=transform)
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=4)

    model = SwinUNet().to(device)
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    # Use ReduceLROnPlateau scheduler
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.1, verbose=True)
    criterion = HybridLoss().to(device)

    for epoch in range(50):
        model.train()
        epoch_loss = 0
        for ct, mri in train_loader:
            ct, mri = ct.to(device), mri.to(device)
            optimizer.zero_grad()
            output = model(ct)
            loss = criterion(output, mri)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        avg_loss = epoch_loss / len(train_loader)
        scheduler.step(avg_loss)
        print(f"Epoch {epoch+1}/50 - Loss: {avg_loss:.4f}")

        # Save the model every 5 epochs
        if (epoch+1) % 5 == 0:
            model_save_path = f"/content/drive/MyDrive/FYP (Final Year Project)/swinunet_epoch{epoch+1}.pth"
            torch.save(model.state_dict(), model_save_path)
            print(f"Model saved to {model_save_path}")

# ------------------------------
# 7. Evaluation Function (Using Test Dataset)
# ------------------------------
def evaluate(model_path):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    transform = Compose([Resize((224, 224)), ToTensor()])
    test_dir = "/content/drive/MyDrive/FYP (Final Year Project)/Datasets/1/CT-MRI Train Test Split Dataset/CT-MRI/test"
    test_dataset = CTMRIDataset(test_dir, transform=transform)
    model = SwinUNet().to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    # Use the first sample from the test dataset for evaluation
    ct, mri = test_dataset[0]
    ct_tensor = ct.unsqueeze(0).to(device)

    with torch.no_grad():
        gen_mri = model(ct_tensor).squeeze().cpu().numpy()

    # Prepare tensors for metric calculations (ensure proper shape)
    gen_mri_tensor = torch.tensor(gen_mri).unsqueeze(0).unsqueeze(0)
    mri_tensor = mri.unsqueeze(0)

    psnr = kornia.metrics.psnr(gen_mri_tensor, mri_tensor, max_val=1.0)
    ssim = kornia.metrics.ssim(gen_mri_tensor, mri_tensor, window_size=11)

    # Visualization
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    ax[0].imshow(ct.squeeze(), cmap='gray')
    ax[0].set_title("Input CT")
    ax[1].imshow(gen_mri, cmap='gray')
    ax[1].set_title(f"Generated MRI\nPSNR: {psnr:.2f} dB, SSIM: {ssim:.4f}")
    ax[2].imshow(mri.squeeze(), cmap='gray')
    ax[2].set_title("Ground Truth MRI")
    plt.show()

# ------------------------------
# 8. Main Execution
# ------------------------------
if __name__ == "__main__":
    train()
    evaluate("/content/drive/MyDrive/FYP (Final Year Project)/swinunet_epoch50.pth")
